In [2]:
from google.colab import files
import pandas as pd
import json

uploaded = files.upload()

Saving cases.csv to cases.csv


In [3]:
cases_df = pd.read_csv(
    "cases.csv"
)

cases_df.head()

,case_id,no_perkara,tanggal,pasal,pihak,ringkasan_fakta,argumen_hukum,jumlah_kata,text_full
0,1,121/Pid.Sus-TPK/2025/PN Bdg,07 Agustus 2025,"Pasal 1, Pasal 11, Pasal 126, Pasal 13, Pasal ...",RIFIAL LUTFHI,Putusan Nomor 121/Pid.Sus-TPK/2025/PN Bdg. P U...,Menimbang bahwa Terdakwa didakwa berdasarkan s...,258360,Putusan Nomor 121/Pid.Sus-TPK/2025/PN Bdg. P U...
1,2,142/Pid.Sus-TPK/2025/PN Bdg,11 September 2025,"Pasal 1, Pasal 11, Pasal 12, Pasal 13, Pasal 1...",Ir. SAIN JUNAEDI Bin KUMPUL. 2,P U T U S A N Nomor 142/Pid.Sus-TPK/2025/PN Bd...,"Menimbang, bahwa Penasihat Hukum Para Terdakwa...",65354,P U T U S A N Nomor 142/Pid.Sus-TPK/2025/PN Bd...
2,3,143/Pid.Sus-TPK/2025/PN Bdg,11 September 2025,"Pasal 1, Pasal 11, Pasal 12, Pasal 13, Pasal 1...","GUNTUR RAHMATULLAH, S.Pd Bin Alm. H. SARDIN 2",P U T U S A N Nomor 143/Pid.Sus-TPK/2025/PN Bd...,"Menimbang, bahwa Penasihat Hukum Terdakwa memo...",63725,P U T U S A N Nomor 143/Pid.Sus-TPK/2025/PN Bd...
3,4,144/Pid.Sus-TPK/2025/PN Bdg,11 September 2025,"Pasal 1, Pasal 11, Pasal 12, Pasal 13, Pasal 1...",MAULANA SOFYAN ARIFIN Bin (Alm) ABDUL HAMID 2,P U T U S A N Nomor 144/Pid.Sus-TPK/2025/PN Bd...,"Menimbang, bahwa Terdakwa diajukan ke persidan...",64134,P U T U S A N Nomor 144/Pid.Sus-TPK/2025/PN Bd...
4,5,26/Pid.Sus-TPK/2026/PN Bdg,17 September 2025,"Pasal 1, Pasal 10, Pasal 145, Pasal 16, Pasal ...",SETIA LIA ANTIKA; 2,Putusan Nomor 26/Pid.Sus-TPK/2026/PN Bdg P U T...,"Menimbang, bahwa Terdakwa dihadapkan ke depan ...",51352,Putusan Nomor 26/Pid.Sus-TPK/2026/PN Bdg P U T...


Ekstrak Solusi dari Semua Putusan

In [4]:
import re

def extract_solution(text):

    text = str(text)

    start = text.find("Menyatakan")

    if start == -1:
        return None

    # berhenti sebelum barang bukti
    end = text.find("5. Menyatakan barang bukti")

    if end == -1:
        end = text.find("Menyatakan barang bukti")

    if end == -1:
        end = text.find("Demikian")

    if end == -1:
        return None

    solusi = text[start:end]

    solusi = re.sub(r"\s+", " ", solusi)

    return solusi.strip()

Apply

In [5]:
cases_df["solusi_text"] = cases_df["text_full"].apply(
    extract_solution
)

Cek Hasil

In [6]:
print(
    cases_df.iloc[0]["solusi_text"][:2000]
)

Menyatakan terdakwa RIFIAL LUTFHI telah terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana “sebagai orang yang melakukan, yang turut serta melakukan perbuatan secara melawan hukum melakukan dengan tujuan menguntungkan diri sendiri atau orang lain atau suatu korporasi, menyalahgunakan kewenangan, kesempatan, atau sarana yang ada padanya karena jabatan atau kedudukan yang dapat merugikan keuangan negara atau perekonomian negara” melanggar Pasal 3 Jo Pasal 18 ayat (1) Undang Undang Nomor 31 tahun 1999 tentang Pemberantasan Tindak Pidana Korupsi sebagaimana telah diubah dengan Undang Undang Nomor 20 tahun 2001 tentang Perubahan Atas Undang Undang Nomor 31 tahun 1999 tentang Pemberantasan Tindak Pidana Korupsi Jo 64 ayat (1) KUHP Jo Pasal 55 ayat (1) Ke-1 KUHP sebagaimana dakwaan Kesatu Subsidair Penuntut Umum; Halaman 2 Putusan Nomor 121/Pid.Sus-TPK/2025/PN Bdg. 2. Menjatuhkan pidana terhadap terdakwa RIFIAL LUTFHI dengan pidana penjara selama 8 (delapan) tahun dan 6 (enam

Cek Missing

In [7]:
cases_df["solusi_text"].isna().sum()

np.int64(0)

TF-IDF

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
vectorizer = TfidfVectorizer(
    max_features=5000
)

tfidf_matrix = vectorizer.fit_transform(
    cases_df["ringkasan_fakta"]
)

Retrieve Cases

In [12]:
def retrieve_cases(query, top_k=5):

    query_vector = vectorizer.transform(
        [query]
    )

    similarities = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    top_idx = similarities.argsort()[-top_k:][::-1]

    hasil = cases_df.iloc[
        top_idx
    ].copy()

    hasil["similarity"] = similarities[
        top_idx
    ]

    return hasil

Predict Outcome

In [13]:

def predict_outcome(query, top_k=5):

    hasil = retrieve_cases(
        query,
        top_k
    )

    best_case = hasil.iloc[0]

    case_id = best_case["case_id"]

    solusi = cases_df.loc[

        cases_df["case_id"] == case_id,

        "solusi_text"

    ].values[0]

    return {

        "query": query,

        "top_5_case_ids":
        hasil["case_id"].tolist(),

        "predicted_solution":
        solusi

    }

Contoh Prediksi

In [14]:
hasil = predict_outcome(
    "korupsi dana desa"
)

print("TOP CASES:")
print(hasil["top_5_case_ids"])

print("\nPREDICTED SOLUTION:")
print(hasil["predicted_solution"][:2000])

TOP CASES:
[3, 2, 16, 9, 6]

PREDICTED SOLUTION:
Menyatakan Terdakwa GUNTUR RAHMATULLAH, S.Pd Bin Alm. H. SARDIN tidak terbukti secara sah dan meyakinkan bersalah melakukan perbuatan “secara bersama-sama melakukan tindak pidana korupsi”, sebagaimana dalam Dakwaan Primair Penuntut Umum Pasal 2 ayat (1) Jo Pasal 18 Undang-Undang Nomor 3l Tahun 1999 tentang Pemberantasan Tindak Pidana sebagaimana telah diubah dengan Undang Undang Nomor 20 Tahun 200l tentang Perubahan atas Undang-Undang Nomor 3l Tahun 1999 tentang Halaman 2 Put No 143 /Pid.Sus-TPK/2025/PN.Bdg Pemberantasan Tindak Pidana Korupsi Jo Pasal 55 ayat (1) ke-1 KUHP yang pengacuannya diubah menjadi Pasal 603 jo Pasal 20 huruf c Undang-Undang Nomor 1 tahun 2023 tentang Kitab Undang-Undang Hukum Pidana sebagaimana telah diubah dengan Undang-Undang Nomor 1 Tahun 2026 tentang Penyesuaian Pidana Jo. Pasal 18 Undang-Undang Republik Indonesia Nomor 31 Tahun 1999 tentang Pemberantasan Tindak Pidana Korupsi sebagaimana telah diubah dan dit

Query Uji

In [15]:
queries = [
    "korupsi dana desa",
    "penyalahgunaan anggaran",
    "kredit fiktif bank",
    "penggelapan dana",
    "kerugian negara"
]

Generate Predictions

In [16]:
predictions = []

for i, q in enumerate(
    queries,
    start=1
):

    hasil = predict_outcome(q)

    predictions.append({

        "query_id": i,

        "query": q,

        "predicted_solution":
        hasil["predicted_solution"],

        "top_5_case_ids":
        str(
            hasil["top_5_case_ids"]
        )

    })

predictions_df = pd.DataFrame(
    predictions
)

predictions_df.head()

,query_id,query,predicted_solution,top_5_case_ids
0,1,korupsi dana desa,"Menyatakan Terdakwa GUNTUR RAHMATULLAH, S.Pd B...","[3, 2, 16, 9, 6]"
1,2,penyalahgunaan anggaran,Menyatakan Terdakwa MOHAMAD BASYIR IDRIS terbu...,"[40, 39, 38, 37, 36]"
2,3,kredit fiktif bank,"Menyatakan Terdakwa ANDRI ACTOVAN, A.Md. tidak...","[7, 21, 40, 39, 36]"
3,4,penggelapan dana,Menyatakan Terdakwa MOHAMAD BASYIR IDRIS terbu...,"[40, 39, 38, 37, 36]"
4,5,kerugian negara,"Menyatakan terdakwa CAKRA ISKANDAR, S.E. tidak...","[21, 3, 13, 2, 9]"


Simpan Predictions

In [17]:
predictions_df.to_csv(

    "predictions.csv",

    index=False,

    encoding="utf-8-sig"

)

print(
    "predictions.csv berhasil dibuat"
)

predictions.csv berhasil dibuat
